In [19]:
import plotly.graph_objects as go
import pandas as pd
from distinctipy import distinctipy
import numpy as np

import os

In [20]:
files = (
    ("48h", "72h"),
    ("72h", "96h"),
    ("96h", "120h"),
    ("120h", "144h"),
)

folders = [
    "sankey/annotation_cell_type/",
    "sankey/annotation_hard/",
]

In [ ]:
def create_unified_sankey(folder, min_flow_threshold=1e-6, ignore_celltypes=None, keep_biggest=1):
    
    # Collect all unique cell types across all timepoints
    all_cell_types = set()
    transition_data = []
    
    # Load all transition matrices
    for t1, t2 in files:
        ot_file = f"{folder}/ot_{t1}_{t2}.csv"
        try:
            df = pd.read_csv(ot_file, index_col=0)
            all_cell_types.update(df.index)
            all_cell_types.update(df.columns)
            transition_data.append((t1, t2, df))
            print(f"Loaded {ot_file}: {df.shape[0]} → {df.shape[1]} cell types")
        except FileNotFoundError:
            print(f"File {ot_file} not found, skipping...")
    
    # Sort cell types alphabetically for consistent coloring
    sorted_cell_types = sorted(all_cell_types)
    print(f"Found {len(sorted_cell_types)} unique cell types across all timepoints")
    print(f"Cell types (alphabetical): {sorted_cell_types}")
    
    # Generate distinct colors for each cell type using distinctipy
    celltype_colors = distinctipy.get_colors(len(sorted_cell_types), pastel_factor=0.3, rng=0)
    celltype_color_map = {}
    
    for i, celltype in enumerate(sorted_cell_types):
        # Convert RGB to RGBA string
        r, g, b = celltype_colors[i]
        celltype_color_map[celltype] = f"rgba({int(r*255)},{int(g*255)},{int(b*255)},0.8)"
    
    print(f"Generated {len(celltype_color_map)} distinct colors for cell types")
    
    # Function to format long labels with line breaks
    def format_label(timepoint, celltype, max_chars=20):
        """Format node labels with line breaks for long cell types."""
        if len(celltype) > max_chars:
            # Try to break at natural word boundaries
            words = celltype.split()
            if len(words) > 1:
                # Split into two lines at word boundary
                mid_point = len(words) // 2
                line1 = " ".join(words[:mid_point])
                line2 = " ".join(words[mid_point:])
                return f"{timepoint}<br>{line1}<br>{line2}"
            else:
                # Single long word - break at character boundary
                mid_point = len(celltype) // 2
                line1 = celltype[:mid_point]
                line2 = celltype[mid_point:]
                return f"{timepoint}<br>{line1}<br>{line2}"
        else:
            return f"{timepoint}<br>{celltype}"
    
    # Create nodes for each cell type at each timepoint with spacing
    timepoints = ["48h", "72h", "96h", "120h", "144h"]
    all_nodes = []
    node_mapping = {}  # (timepoint, celltype) -> node_index
    
    # Add spacing between timepoints by creating empty spacer nodes
    for tp_idx, timepoint in enumerate(timepoints):
        # Add spacer nodes between timepoints (except before first)
        if tp_idx > 0:
            # Add invisible spacer nodes
            for spacer in range(2):  # Add 2 spacer nodes for more separation
                spacer_label = f"spacer_{tp_idx}_{spacer}"
                all_nodes.append(spacer_label)
        
        # Add actual cell type nodes for this timepoint
        for celltype in sorted_cell_types:
            node_label = format_label(timepoint, celltype)
            node_index = len(all_nodes)
            all_nodes.append(node_label)
            node_mapping[(timepoint, celltype)] = node_index
    
    print(f"Created {len(all_nodes)} nodes")
    
    # Collect all flows
    source_indices = []
    target_indices = []
    flow_values = []
    
    for t1, t2, df in transition_data:
        print(f"Processing {t1} → {t2}")
        
        for target_celltype in df.columns:
            l = []
            sl = []
            for source_celltype in df.index:
                if source_celltype in ignore_celltypes or target_celltype in ignore_celltypes:
                    continue
                flow_value = df.loc[source_celltype, target_celltype]
                l.append(flow_value)
                sl.append(source_celltype)
            vals = np.array(sl)[np.argsort(l)[::-1]][:keep_biggest]
            print(target_celltype, vals)

            for source_celltype in df.index:
                if ignore_celltypes and (source_celltype in ignore_celltypes or target_celltype in ignore_celltypes):
                    continue

                flow_value = df.loc[source_celltype, target_celltype]
                
                if flow_value > min_flow_threshold or source_celltype in vals:
                    # Get node indices
                    source_idx = node_mapping[(t1, source_celltype)]
                    target_idx = node_mapping[(t2, target_celltype)]
                    
                    source_indices.append(source_idx)
                    target_indices.append(target_idx)
                    flow_values.append(flow_value)
    
    print(f"Total flows above threshold: {len(flow_values)}")
    if flow_values:
        print(f"Flow values range: {min(flow_values):.2e} to {max(flow_values):.2e}")
    
    # Assign colors to nodes based on cell type (ignoring timepoint and handling spacers)
    node_colors = []
    for node_label in all_nodes:
        if node_label.startswith("spacer_"):
            # Make spacer nodes transparent/invisible
            node_colors.append("rgba(255,255,255,0)")
        else:
            # Extract cell type from formatted label (everything after <br> tags)
            # Format is "timepoint<br>celltype" or "timepoint<br>line1<br>line2"
            parts = node_label.split("<br>")
            if len(parts) >= 2:
                # Reconstruct cell type from parts (skip timepoint)
                celltype = " ".join(parts[1:])
                node_colors.append(celltype_color_map.get(celltype, "lightgray"))
            else:
                # Fallback for any unexpected format
                node_colors.append("lightgray")
    
    # Create Sankey diagram
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=20,
            thickness=30,
            line=dict(color="black", width=1),
            label=all_nodes,
            color=node_colors
        ),
        link=dict(
            source=source_indices,
            target=target_indices,
            value=flow_values,
            color="rgba(100,100,100,0.3)"
        )
    )])
    
    fig.update_layout(
        title="Unified Optimal Transport: Complete Developmental Trajectory<br><sub>Colors indicate cell types (consistent across timepoints)</sub>",
        font_size=20,
        width=2400,
        height=1600,
        title_font_size=22,
        font_family="Arial",
        plot_bgcolor='white',
        paper_bgcolor='white'
    )
    
    # Print color mapping for reference
    print("\nColor mapping for cell types:")
    for celltype, color in celltype_color_map.items():
        print(f"  {celltype}: {color}")
    
    return fig

In [22]:
for folder in folders[1:]:
    unified_fig = create_unified_sankey(folder, min_flow_threshold=1e-5, ignore_celltypes=["Unconfident","Unassigned"])
    unified_fig.show()
    unified_fig.write_html(f"figures/sankey_unified_all_transitions_{folder.strip('/').replace('/', '_')}.html")
    unified_fig.write_image(f"figures/sankey_unified_all_transitions_{folder.strip('/').replace('/', '_')}.svg")

Loaded sankey/annotation_hard//ot_48h_72h.csv: 2 → 3 cell types
Loaded sankey/annotation_hard//ot_72h_96h.csv: 3 → 7 cell types
Loaded sankey/annotation_hard//ot_96h_120h.csv: 7 → 12 cell types
Loaded sankey/annotation_hard//ot_120h_144h.csv: 12 → 14 cell types
Found 28 unique cell types across all timepoints
Cell types (alphabetical): ['Anterior Epiblast', 'Anterior Mid Primitive Streak', 'Anterior Notochord', 'Anterior Somitic Mesoderm', 'Anterior epiblast/Neuroectoderm', 'Anterior/Head Mesoderm', 'Branchial Arches', 'Early Definitive Endoderm', 'Early Epiblast', 'Early Mesoderm (Prospective Paraxial Mesoderm)', 'Early Mesoderm + Extraembryonic Mesoderm', 'Early Organiser', 'Early Primitive Streak', 'Early differentiating neurons', 'Epiblast', 'Floor Plate + Pharyngeal Mesendoderm', 'Foregut', 'Gut/Endoderm', 'Heart Progenitors', 'Hematoendothelial Progenitors', 'Hematoendothelial progenitors + Extraembryonic mesoderm + Mesendoderm', 'Late Posterior PS / Early Caudal epiblast', 'Mid 

In [23]:
interest = set([
    "Pluripotency",
    "Early Epiblast",
    "Early Primitive Streak",
    "Middle Primitive Streak",
    "Epiblast",
    "Anterior Epiblast",
    "Late Posterior PS / Early Caudal epiblast",
    "Neural tube/Anterior Spinal Cord",
    "Anterior epiblast/Neuroectoderm",
    "Early differentiating neurons",
    "Hindbrain/Rhombomeres"
])

for folder in folders[1:]:

    l = []
    for file in os.listdir(folder):
        df = pd.read_csv(os.path.join(folder, file))
        l += list(df.index.values)
        l += list(df.columns.values)        
    l = set(l)

    ignored_celltypes = l - interest

    unified_fig = create_unified_sankey(folder, min_flow_threshold=1e-8, ignore_celltypes=list(ignored_celltypes))
    unified_fig.show()
    unified_fig.write_html(f"figures/sankey_unified_all_transitions_{folder.strip('/').replace('/', '_')}_reduced.html")
    unified_fig.write_image(f"figures/sankey_unified_all_transitions_{folder.strip('/').replace('/', '_')}_reduced.svg")

Loaded sankey/annotation_hard//ot_48h_72h.csv: 2 → 3 cell types
Loaded sankey/annotation_hard//ot_72h_96h.csv: 3 → 7 cell types
Loaded sankey/annotation_hard//ot_96h_120h.csv: 7 → 12 cell types
Loaded sankey/annotation_hard//ot_120h_144h.csv: 12 → 14 cell types
Found 28 unique cell types across all timepoints
Cell types (alphabetical): ['Anterior Epiblast', 'Anterior Mid Primitive Streak', 'Anterior Notochord', 'Anterior Somitic Mesoderm', 'Anterior epiblast/Neuroectoderm', 'Anterior/Head Mesoderm', 'Branchial Arches', 'Early Definitive Endoderm', 'Early Epiblast', 'Early Mesoderm (Prospective Paraxial Mesoderm)', 'Early Mesoderm + Extraembryonic Mesoderm', 'Early Organiser', 'Early Primitive Streak', 'Early differentiating neurons', 'Epiblast', 'Floor Plate + Pharyngeal Mesendoderm', 'Foregut', 'Gut/Endoderm', 'Heart Progenitors', 'Hematoendothelial Progenitors', 'Hematoendothelial progenitors + Extraembryonic mesoderm + Mesendoderm', 'Late Posterior PS / Early Caudal epiblast', 'Mid 